<a href="https://colab.research.google.com/github/soumensen411/Deep-Learning-Model/blob/main/RNN/Automatic_sentence_completion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

In [7]:
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Deep Learning/RNN Project/automatic sentence completion project/qoute_dataset.csv')

In [8]:
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [9]:
df = df['quote'].str.lower()

In [10]:
df = df.drop(columns=['Author'])

In [11]:
df.head()

,quote
0,“the world as we have created it is a process ...
1,"“it is our choices, harry, that show what we t..."
2,“there are only two ways to live your life. on...
3,"“the person, be it gentleman or lady, who has ..."
4,"“imperfection is beauty, madness is genius and..."


In [12]:
import string
translator = str.maketrans('', '', string.punctuation)
df = df.apply(lambda x:x.translate(translator))

In [13]:
df.head()

,quote
0,“the world as we have created it is a process ...
1,“it is our choices harry that show what we tru...
2,“there are only two ways to live your life one...
3,“the person be it gentleman or lady who has no...
4,“imperfection is beauty madness is genius and ...


In [14]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [15]:
vocab_size = 10000
tokenizer = Tokenizer(num_words=vocab_size)
tokenizer.fit_on_texts(df)

In [16]:
word_index = tokenizer.word_index
list(word_index.items())[:10]

[('the', 1),
 ('you', 2),
 ('to', 3),
 ('and', 4),
 ('a', 5),
 ('i', 6),
 ('is', 7),
 ('of', 8),
 ('that', 9),
 ('it', 10)]

In [17]:
sequence = tokenizer.texts_to_sequences(df)

In [18]:
for i in range(3):
  print(df[i])

“the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”
“it is our choices harry that show what we truly are far more than our abilities”
“there are only two ways to live your life one is as though nothing is a miracle the other is as though everything is a miracle”


In [19]:
for i in range(3):
  print(sequence[i])

[713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12, 809, 104, 752, 70, 2461]
[947, 7, 70, 871, 373, 9, 433, 21, 19, 465, 14, 294, 52, 54, 70, 3676]
[1337, 14, 53, 201, 714, 3, 81, 15, 36, 37, 7, 29, 329, 93, 7, 5, 1157, 1, 101, 7, 29, 329, 126, 7, 5, 3677]


In [20]:
X = []
y = []
for seq in sequence:
  for i in range(1,len(seq)):
    input_seq = seq[:i]
    out_seq = seq[i]
    X.append(input_seq)
    y.append(out_seq)

In [21]:
len(X)

85271

In [22]:
len(y)

85271

In [23]:
max_len = max(len(i) for i in X)

In [24]:
print(max_len)

745


In [25]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [26]:
X_padded = pad_sequences(X, maxlen=max_len, padding='pre')

In [27]:
print(X_padded.shape)
print(X_padded[0])

(85271, 745)
[  0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0  

In [28]:
y = np.array(y)

In [29]:
X_padded.shape

(85271, 745)

In [30]:
y.shape

(85271,)

In [31]:
from tensorflow.keras.utils import to_categorical
y_cat = to_categorical(y,num_classes = vocab_size)

In [32]:
y_cat.shape

(85271, 10000)

In [33]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,SimpleRNN,LSTM,Dense

In [34]:
embedding_dim = 50
rnn_units = 128


In [35]:
rnn_model = Sequential([
    Embedding(input_dim=vocab_size,output_dim=embedding_dim),
    SimpleRNN(units=rnn_units),
    Dense(units=vocab_size,activation='softmax')
])

In [36]:
rnn_model.compile(
    optimizer = 'adam',
    loss = 'categorical_crossentropy',
    metrics = ['accuracy']
)

In [37]:
rnn_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

## Train the SimpleRNN Model

In [41]:
rnn_model.compile(
    optimizer = 'adam',
    loss = 'categorical_crossentropy',
    metrics = ['accuracy']
)
history_rnn = rnn_model.fit(X_padded, y_cat, epochs=10,batch_size=128,validation_split=0.1, verbose=1)

Epoch 1/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 47s 70ms/step - accuracy: 0.0438 - loss: 6.7120 - val_accuracy: 0.0570 - val_loss: 6.5702
Epoch 2/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 39s 64ms/step - accuracy: 0.0767 - loss: 6.1229 - val_accuracy: 0.0879 - val_loss: 6.3324
Epoch 3/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 38s 64ms/step - accuracy: 0.1015 - loss: 5.7588 - val_accuracy: 0.1023 - val_loss: 6.3022
Epoch 4/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 39s 64ms/step - accuracy: 0.1169 - loss: 5.4600 - val_accuracy: 0.1034 - val_loss: 6.2978
Epoch 5/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 39s 64ms/step - accuracy: 0.1308 - loss: 5.1966 - val_accuracy: 0.1096 - val_loss: 6.3846
Epoch 6/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 39s 64ms/step - accuracy: 0.1431 - loss: 4.9569 - val_accuracy: 0.1099 - val_loss: 6.4201
Epoch 7/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 39s 64ms/step - accuracy: 0.1587 - loss: 4.7345 - val_accuracy: 0.1102 - val_loss: 6.4940
Epoch 8/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 38s 64ms/step - accuracy: 0.1754 - loss: 4.5246 - 

In [42]:
print(history_rnn.history)

{'accuracy': [0.043847646564245224, 0.07671058177947998, 0.10154671967029572, 0.11685756593942642, 0.1307871788740158, 0.14307494461536407, 0.15867245197296143, 0.17536452412605286, 0.1973469853401184, 0.21397390961647034], 'loss': [6.712038993835449, 6.122928619384766, 5.758785724639893, 5.460005760192871, 5.196591854095459, 4.956906795501709, 4.734501838684082, 4.524611949920654, 4.3275885581970215, 4.173712730407715], 'val_accuracy': [0.05698874220252037, 0.08794558793306351, 0.10225141048431396, 0.10342401266098022, 0.10963883996009827, 0.10987336188554764, 0.1102251410484314, 0.10881800949573517, 0.10541744530200958, 0.10518292337656021], 'val_loss': [6.570178985595703, 6.332424163818359, 6.302224159240723, 6.297768592834473, 6.384556770324707, 6.420111656188965, 6.494010925292969, 6.568122386932373, 6.6401286125183105, 6.720135688781738]}


## Train the LSTM Model

In [1]:
lstm_model = Sequential([
    Embedding(input_dim=vocab_size,output_dim=embedding_dim),
    LSTM(units=rnn_units),
    Dense(units=vocab_size,activation='softmax')
])
lstm_model.compile(
    optimizer = 'adam',
    loss = 'categorical_crossentropy',
    metrics = ['accuracy']
)
history_lstm = lstm_model.fit(X_padded, y_cat, epochs=100,batch_size = 128,validation_split=0.1, verbose=1)

NameError: name 'Sequential' is not defined

In [44]:
print(history_lstm.history)

{'accuracy': [0.03834877535700798, 0.05678693950176239, 0.07908213138580322, 0.09619118273258209, 0.10748863220214844, 0.1184081956744194, 0.1278553158044815, 0.13661180436611176, 0.14285342395305634, 0.15055444836616516], 'loss': [6.767091751098633, 6.330166339874268, 6.0604777336120605, 5.839702606201172, 5.659722328186035, 5.486597537994385, 5.3246169090271, 5.176711559295654, 5.038149833679199, 4.905960559844971], 'val_accuracy': [0.04303470999002457, 0.0633208230137825, 0.08501406759023666, 0.09509849548339844, 0.1011960580945015, 0.1044793650507927, 0.1094043180346489, 0.10870075225830078, 0.11198405176401138, 0.1159709170460701], 'val_loss': [6.683888912200928, 6.579281806945801, 6.457019805908203, 6.42842960357666, 6.421191692352295, 6.415761470794678, 6.4355363845825195, 6.462474822998047, 6.51107120513916, 6.558382987976074]}
